In [1]:
!pip install -qU  langchain-community langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.8/84.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 489.1/489.1 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [2]:
import langchain
langchain.__version__

'1.2.0'

## SQLDatabase Toolkit

This will help you get started with the SQL Database toolkit. For detailed documentation of all SQLDatabaseToolkit features and configurations head to the API reference.

Tools within the SQLDatabaseToolkit are designed to interact with a SQL database.

A common application is to enable agents to answer questions using data in a relational database, potentially in an iterative fashion (e.g., recovering from errors).

### Instantiation
The **`SQLDatabaseToolkit`** toolkit requires:
*   a **SQLDatabase** object;
*   a LLM or chat model (for instantiating the **QuerySQLCheckerTool** tool).

Below, we instantiate the toolkit with these objects. Let’s first create a database object.

This guide uses the example **`Chinook`** database based on these instructions.

Below we will use the requests library to pull the .sql file and create an in-memory SQLite database. Note that this approach is lightweight, but ephemeral and not thread-safe. If you’d prefer, you can follow the instructions to save the file locally as `**Chinook.db**` and instantiate the database via **`db = SQLDatabase.from_uri("sqlite:///Chinook.db")`**.

In [3]:
import sqlite3

import requests
from langchain_community.utilities.sql_database import SQLDatabase
from sqlalchemy import create_engine
from sqlalchemy.pool import StaticPool


def get_engine_for_chinook_db():
    """Pull sql file, populate in-memory database, and create engine."""
    url = "https://raw.githubusercontent.com/lerocha/chinook-database/master/ChinookDatabase/DataSources/Chinook_Sqlite.sql"
    response = requests.get(url)
    sql_script = response.text

    connection = sqlite3.connect(":memory:", check_same_thread=False)
    connection.executescript(sql_script)
    return create_engine(
        "sqlite://",
        creator=lambda: connection,
        poolclass=StaticPool,
        connect_args={"check_same_thread": False},
    )


engine = get_engine_for_chinook_db()

db = SQLDatabase(engine)

In [4]:
import os
from google.colab import userdata

if userdata.get("OPENAI_API_KEY"):
  os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
  os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter API key for OpenAI: ")

In [5]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model_name="gpt-4o-mini",
    temperature=0
    )

### Tools

In [6]:
from langchain_community.agent_toolkits.sql.toolkit import SQLDatabaseToolkit

toolkit = SQLDatabaseToolkit(db=db, llm=llm)

In [7]:
toolkit.get_tools()

[QuerySQLDatabaseTool(description="Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is not correct, an error message will be returned. If an error is returned, rewrite the query, check the query, and try again. If you encounter an issue with Unknown column 'xxxx' in 'field list', use sql_db_schema to query the correct table fields.", db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7e2a5e1e91f0>),
 InfoSQLDatabaseTool(description='Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables. Be sure that the tables actually exist by calling sql_db_list_tables first! Example Input: table1, table2, table3', db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7e2a5e1e91f0>),
 ListSQLDatabaseTool(db=<langchain_community.utilities.sql_database.SQLDatabase object at 0x7e2a5e1e91f0>),
 QuerySQLCheckerTool(description='Use this tool to double check

In [8]:
from langchain_community.tools.sql_database.tool import (
    InfoSQLDatabaseTool,
    ListSQLDatabaseTool,
    QuerySQLCheckerTool,
    QuerySQLDatabaseTool,
)

### Use within an agent

In [9]:
from langchain_classic import hub

prompt_template = hub.pull("langchain-ai/sql-agent-system-prompt")

assert len(prompt_template.messages) == 1
print(prompt_template.input_variables)

['dialect', 'top_k']


In [10]:
system_message = prompt_template.format(dialect="SQLite", top_k=5)

In [11]:
from langchain.agents import create_agent

agent = create_agent(llm, toolkit.get_tools(), system_prompt=system_message)

In [12]:
example_query = "Which country's customers spent the most?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Which country's customers spent the most?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_huiraEmo5wEkqok7U3PjufrB)
 Call ID: call_huiraEmo5wEkqok7U3PjufrB
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_NyGylyHtlRlXW6JwBLh9QVBm)
 Call ID: call_NyGylyHtlRlXW6JwBLh9QVBm
  Args:
    table_names: Customer
  sql_db_schema (call_1oFAcyfMBftdC4l5adnESff2)
 Call ID: call_1oFAcyfMBftdC4l5adnESff2
  Args:
    table_names: Invoice
================================= Tool Message =================================
Name: sql_db_schema


CREATE TABLE "Invoice" (
	"Inv

In [13]:
example_query = "Who are the top 3 best selling artists?"

events = agent.stream(
    {"messages": [("user", example_query)]},
    stream_mode="values",
)
for event in events:
    event["messages"][-1].pretty_print()

================================ Human Message =================================

Who are the top 3 best selling artists?
================================== Ai Message ==================================
Tool Calls:
  sql_db_list_tables (call_ktBIgAjN1KEprAD3pReX06Fb)
 Call ID: call_ktBIgAjN1KEprAD3pReX06Fb
  Args:
================================= Tool Message =================================
Name: sql_db_list_tables

Album, Artist, Customer, Employee, Genre, Invoice, InvoiceLine, MediaType, Playlist, PlaylistTrack, Track
================================== Ai Message ==================================
Tool Calls:
  sql_db_schema (call_Max6dpncqD8JbJ3kTwGvDchC)
 Call ID: call_Max6dpncqD8JbJ3kTwGvDchC
  Args:
    table_names: Artist
  sql_db_schema (call_gmx59wrJjQcEhgwxaHKNqyIm)
 Call ID: call_gmx59wrJjQcEhgwxaHKNqyIm
  Args:
    table_names: InvoiceLine
  sql_db_schema (call_ifrCsjaXfpnuXjvJijPoGAFP)
 Call ID: call_ifrCsjaXfpnuXjvJijPoGAFP
  Args:
    table_names: Track
==============